In [ ]:

import pyodbc
import ollama

# --------------------------------------------------
# 1. SQL SERVER CONNECTION
# --------------------------------------------------

connection = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=DESKTOP-NSQVUD1;"
    "DATABASE=orderssystem;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

cursor = connection.cursor()


# --------------------------------------------------
# 2. FUNCTION TO EXECUTE SQL AND FORMAT RESULTS
# --------------------------------------------------

def get_data(query):

    cursor.execute(query)

    columns = [column[0] for column in cursor.description]

    rows = cursor.fetchall()

    result = []

    for row in rows:

        record = {}

        for i, column in enumerate(columns):
            record[column] = row[i]

        result.append(record)

    return result


def format_data(data):

    if not data:
        return "No records found."

    lines = []

    for record in data:

        lines.append(
            " | ".join(
                f"{key}: {value}"
                for key, value in record.items()
            )
        )

    return "\n".join(lines)


# --------------------------------------------------
# 3. CATEGORY-WISE TOTAL EXPENSE ANALYSIS
# --------------------------------------------------

category_expenses = get_data("""
SELECT
    ec.categoryid,
    ec.categoryname,
    parent.categoryname AS parentcategory,
    COUNT(e.expenseid) AS expensecount,
    ISNULL(SUM(e.amount), 0) AS totalexpense,
    ISNULL(AVG(e.amount), 0) AS averageexpense,
    ISNULL(MAX(e.amount), 0) AS highestexpense,
    ISNULL(MIN(e.amount), 0) AS lowestexpense

FROM expensecategory ec

LEFT JOIN expensecategory parent
    ON ec.parentcategoryid = parent.categoryid

LEFT JOIN expense e
    ON ec.categoryid = e.categoryid

GROUP BY
    ec.categoryid,
    ec.categoryname,
    parent.categoryname

ORDER BY totalexpense DESC;
""")


# --------------------------------------------------
# 4. MONTHLY CATEGORY-WISE EXPENSE ANALYSIS
# --------------------------------------------------

monthly_expenses = get_data("""
SELECT
    ec.categoryname,
    YEAR(e.expensedate) AS expenseyear,
    MONTH(e.expensedate) AS expensemonth,
    COUNT(e.expenseid) AS expensecount,
    SUM(e.amount) AS totalexpense

FROM expense e

INNER JOIN expensecategory ec
    ON e.categoryid = ec.categoryid

GROUP BY
    ec.categoryname,
    YEAR(e.expensedate),
    MONTH(e.expensedate)

ORDER BY
    expenseyear,
    expensemonth,
    totalexpense DESC;
""")


# --------------------------------------------------
# 5. CATEGORY-WISE BUDGET VS ACTUAL EXPENSE
# --------------------------------------------------

budget_analysis = get_data("""
WITH ActualExpenses AS
(
    SELECT
        categoryid,
        YEAR(expensedate) AS expenseyear,
        MONTH(expensedate) AS expensemonth,
        SUM(amount) AS actualexpense

    FROM expense

    GROUP BY
        categoryid,
        YEAR(expensedate),
        MONTH(expensedate)
)

SELECT
    ec.categoryname,
    b.budgetyear,
    b.budgetmonth,
    b.budgetamount,

    ISNULL(a.actualexpense, 0) AS actualexpense,

    b.budgetamount -
        ISNULL(a.actualexpense, 0) AS budgetbalance,

    CASE
        WHEN ISNULL(a.actualexpense, 0) > b.budgetamount
            THEN 'Over Budget'

        WHEN ISNULL(a.actualexpense, 0) < b.budgetamount
            THEN 'Within Budget'

        ELSE 'Budget Fully Utilized'
    END AS budgetstatus

FROM expensebudget b

INNER JOIN expensecategory ec
    ON b.categoryid = ec.categoryid

LEFT JOIN ActualExpenses a
    ON b.categoryid = a.categoryid
    AND b.budgetyear = a.expenseyear
    AND b.budgetmonth = a.expensemonth

ORDER BY
    b.budgetyear,
    b.budgetmonth,
    ec.categoryname;
""")


# --------------------------------------------------
# 6. CATEGORY-WISE PAYMENT ANALYSIS
# --------------------------------------------------

payment_analysis = get_data("""
WITH PaymentTotals AS
(
    SELECT
        expenseid,
        SUM(
            CASE
                WHEN status = 'Paid' THEN paidamount
                ELSE 0
            END
        ) AS totalpaid

    FROM payment

    GROUP BY expenseid
)

SELECT
    ec.categoryname,

    SUM(e.amount) AS totalexpense,

    SUM(ISNULL(p.totalpaid, 0)) AS totalpaid,

    SUM(
        e.amount - ISNULL(p.totalpaid, 0)
    ) AS outstandingamount

FROM expense e

INNER JOIN expensecategory ec
    ON e.categoryid = ec.categoryid

LEFT JOIN PaymentTotals p
    ON e.expenseid = p.expenseid

GROUP BY ec.categoryname

ORDER BY totalexpense DESC;
""")


# --------------------------------------------------
# 7. HIGHEST EXPENSE CATEGORIES
# --------------------------------------------------

top_categories = get_data("""
SELECT TOP 10
    ec.categoryname,
    SUM(e.amount) AS totalexpense,
    COUNT(e.expenseid) AS expensecount

FROM expense e

INNER JOIN expensecategory ec
    ON e.categoryid = ec.categoryid

GROUP BY ec.categoryname

ORDER BY totalexpense DESC;
""")


# --------------------------------------------------
# 8. CATEGORY-WISE EXPENSE DETAILS
# --------------------------------------------------

expense_details = get_data("""
SELECT TOP 100
    e.expenseid,
    ec.categoryname,
    e.expensedate,
    e.description,
    v.vendorname,
    e.amount

FROM expense e

INNER JOIN expensecategory ec
    ON e.categoryid = ec.categoryid

LEFT JOIN vendor v
    ON e.vendorid = v.vendorid

ORDER BY e.expensedate DESC;
""")


# --------------------------------------------------
# 9. PREPARE DATA FOR OLLAMA
# --------------------------------------------------

all_data = f"""

CATEGORY-WISE EXPENSE ANALYSIS

1. CATEGORY SUMMARY

{format_data(category_expenses)}


2. MONTHLY CATEGORY ANALYSIS

{format_data(monthly_expenses)}


3. BUDGET VS ACTUAL ANALYSIS

{format_data(budget_analysis)}


4. CATEGORY-WISE PAYMENT ANALYSIS

{format_data(payment_analysis)}


5. TOP EXPENSE CATEGORIES

{format_data(top_categories)}


6. EXPENSE DETAILS

{format_data(expense_details)}

"""


# --------------------------------------------------
# 10. CATEGORY ANALYSIS AGENT
# --------------------------------------------------

def category_expense_agent(user_question):

    system_prompt = """
You are an expert Expense Category Analysis Agent.

You analyse company expenses using SQL Server data.

You can answer questions about:

1. Category-wise total expenses.
2. Subcategory-wise expenses.
3. Monthly category expenses.
4. Highest and lowest expense categories.
5. Average expense by category.
6. Budget versus actual expenses.
7. Over-budget categories.
8. Remaining budget.
9. Category-wise payments.
10. Outstanding expenses.
11. Expense trends.
12. Unusual spending patterns.
13. Management recommendations.

IMPORTANT RULES:

- Use only the supplied database information.
- Do not invent financial figures.
- Clearly distinguish actual expenses from payments.
- Outstanding amount is expense amount minus successful payments.
- Do not add payments to expenses when calculating total expenses.
- Do not confuse a parent category with its subcategories.
- If information is unavailable, say so.
- Show financial amounts with two decimal places.
- Explain your findings in simple business language.
- Give practical recommendations.
"""

    user_prompt = f"""
DATABASE INFORMATION:

{all_data}

USER QUESTION:

{user_question}

Analyse the supplied data and provide a clear answer.
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    return response["message"]["content"]


# --------------------------------------------------
# 11. INTERACTIVE AGENT
# --------------------------------------------------

print("\nCATEGORY-WISE EXPENSE ANALYSIS AGENT")
print("-------------------------------------")

while True:

    question = input("\nAsk your question (type exit to stop): ")

    if question.lower().strip() == "exit":
        break

    try:

        answer = category_expense_agent(question)

        print("\nAI ANALYSIS:")
        print("------------------------------")
        print(answer)

    except Exception as e:

        print("Error:", e)


# --------------------------------------------------
# 12. CLOSE CONNECTION
# --------------------------------------------------

cursor.close()
connection.close()

print("\nCategory Expense Agent stopped.")



CATEGORY-WISE EXPENSE ANALYSIS AGENT
-------------------------------------



Ask your question (type exit to stop):  average expenses by category do not explain



AI ANALYSIS:
------------------------------
To calculate the average expenses by category, we need to divide the total expense by the number of expenses for each category. 

However, upon analyzing the data, it appears that the category "Salary" is not a parent category, but rather a child category. The parent category is "Office Supplies". 

To calculate the average expense for "Office Supplies", we need to add the total expenses of all its child categories. 

The total expenses for the child categories of "Office Supplies" are: 999.00 (Telephone) + 1800.00 (Office Supplies) = 2799.00. 

There are 2 expenses for the category "Office Supplies" (Category ID: 2 and Category ID: 3). 

Average expense for "Office Supplies" = Total expenses / Number of expenses = 2799.00 / 2 = 1399.50.



Ask your question (type exit to stop):  highest categories



AI ANALYSIS:
------------------------------
To find the highest expense categories, we need to analyze the data and identify the categories with the highest total expenses.

From the supplied data, we can see that there are two categories with the highest total expenses:

1. Hotel: With a total expense of $24,000.00, Hotel has the highest expense category.
2. Printing: With a total expense of $4,700.00, Printing is the second-highest expense category.

These two categories have the highest total expenses, indicating that Hotel and Printing are the most significant expense categories for the company.

To provide a more detailed analysis, here are the top 3 highest expense categories:

1. Hotel: $24,000.00
2. Printing: $4,700.00
3. Office Supplies: $1,800.00

These categories have the highest total expenses, and it's essential for the company to monitor and manage their expenses closely to ensure they stay within budget.

Recommendation: The company should review and analyze the expense